In [1]:
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import os
import sys
sys.path.append("..")
from model_embeddings import IMAGES_DIR, images
import warnings
import uuid

/home/zab/rag-sandbox/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_DIR = os.path.join(os.path.dirname(os.getcwd()), "clothing_images")
images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR)]

model = None
embeddings = None
client = None

In [3]:
def load_model():
    global model
    if model is None:
        model = SentenceTransformer(
            "jinaai/jina-clip-v2",
            trust_remote_code=True,
            truncate_dim=1024,
            device=device,
            model_kwargs={"torch_dtype":torch.float16}
        )
    else:
        warnings.warn("Model already loaded", UserWarning)

load_model()

/home/zab/.cache/huggingface/modules/transformers_modules/jinaai/jina-clip-implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:140: UserWarning: Flash attention is not installed. Check https://github.com/Dao-AILab/flash-attention?tab=readme-ov-file#installation-and-features for installation instructions, disabling
  warnings.warn(
/home/zab/.cache/huggingface/modules/transformers_modules/jinaai/jina-clip-implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:175: UserWarning: xFormers is not installed. Check https://github.com/facebookresearch/xformers?tab=readme-ov-file#installing-xformers for installation instructions, disabling
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with

In [4]:
def load_embeddings():
    global embeddings
    if embeddings is None:

        embeddings = model.encode(images, batch_size = 16, normalize_embeddings = True, show_progress_bar = True)
    else:
        warnings.warn("Embeddings already loaded", UserWarning)

load_embeddings()

Batches:   2%|▏         | 13/626 [00:14<11:13,  1.10s/it]


KeyboardInterrupt: 

Embeddings that worked: 8, 16

In [ ]:

def client_creation():
    global client
    if not os.path.exists("image_store") and client is None:

        client = QdrantClient(host="localhost", port=6333)

        client.create_collection(
            collection_name="images",
            vectors_config=VectorParams(size=len(embeddings[0]), distance=Distance.COSINE)
        )
        client.upload_points(
            collection_name="images",
            points=[
                PointStruct(id=uuid.uuid4(), vector=embeddings[i], payload={"path": images[i]})
                for i in range(len(images))
            ],
            batch_size=100
        )
    else:
        warnings.warn("Qdrant client already up", UserWarning)

client_creation()

In [ ]:
def search_object(search_query):
    query_embeddings = model.encode(search_query, normalize_embeddings=True, device=device)
    results = client.query_points(collection_name="images", query=query_embeddings, limit=5).points
    return results
search_object("red shirt")